# Chapter 3 — Structured Extraction

**Goal:** Teach the model to extract fields from free text and output valid JSON.

Chapter 2 was about picking one label from a fixed set. This chapter levels up: the model must **generate structured output** with the right fields and values. It's the same fine-tuning technique — only the output format changes.

**Task:** Given a sentence about a person's job, extract `name`, `company`, `role`, `start_date` as JSON.

## 3.1 Load the Base Model

Fresh SmolLM — no prior task training.

In [ ]:
import torch, json, re
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

BASE_MODEL = "HuggingFaceTB/SmolLM-135M"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
if device == "cpu":
    model = model.to(device)

print(f"Model loaded on {device}")

## 3.2 Create Training Data

We generate sentences from templates and produce the corresponding JSON. Each example is `Extract: <sentence>\nJSON:` → ` <json>`.

**Why this format matters:** the `Extract:` prefix tells the model what task to perform, and `JSON:` signals what output format to use. This is the same "prompt engineering" pattern used with large models — we're just baking it into the small model via fine-tuning.

In [ ]:
import random
random.seed(42)

def make_example(sentence, fields):
    prompt = f"Extract: {sentence}\nJSON:"
    completion = " " + json.dumps(fields)
    return {"prompt": prompt, "completion": completion}

names = ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank", "Grace", "Hank",
         "Iris", "Jack", "Kate", "Leo", "Maria", "Nick", "Olivia", "Pat"]
companies = ["Acme Corp", "Globex Inc", "Initech", "Umbrella LLC", "Stark Industries",
             "Wayne Enterprises", "Cyberdyne", "Massive Dynamic"]
roles = ["software engineer", "data scientist", "product manager", "UX designer",
         "devops engineer", "engineering manager", "QA lead", "technical writer"]
months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

raw_data = []

for _ in range(400):
    name = random.choice(names)
    company = random.choice(companies)
    role = random.choice(roles)
    month = random.choice(months)
    year = random.choice(["2020", "2021", "2022", "2023", "2024"])
    start = f"{month} {year}"

    fields = {"name": name, "company": company, "role": role, "start_date": start}

    # Vary the sentence phrasing so the model doesn't memorize a single template
    templates = [
        f"{name} joined {company} as a {role} in {start}",
        f"{name} started working at {company} as a {role} in {start}",
        f"In {start}, {name} became a {role} at {company}",
        f"{company} hired {name} as a {role} in {start}",
        f"{name} has been a {role} at {company} since {start}",
        f"Since {start}, {name} has worked as a {role} for {company}",
    ]
    sentence = random.choice(templates)
    raw_data.append(make_example(sentence, fields))

random.shuffle(raw_data)
split = int(0.85 * len(raw_data))
train_data = raw_data[:split]
eval_data = raw_data[split:]

print(f"Train: {len(train_data)}  |  Eval: {len(eval_data)}")
print(f"\nSample:\nprompt:     {train_data[0]['prompt']}")
print(f"completion: {train_data[0]['completion']}")

## 3.3 Tokenize with Prompt Masking

Same technique as Chapter 2: tokenize prompt and completion separately, mask prompt tokens in labels, append EOS.

In [ ]:
def tokenize(example):
    prompt_tokens = tokenizer(example["prompt"], truncation=True, max_length=192, padding=False)
    completion_tokens = tokenizer(example["completion"], truncation=True, max_length=96, padding=False)

    input_ids = prompt_tokens["input_ids"] + completion_tokens["input_ids"] + [tokenizer.eos_token_id]
    attention_mask = [1] * len(input_ids)
    labels = [-100] * len(prompt_tokens["input_ids"]) + completion_tokens["input_ids"] + [tokenizer.eos_token_id]

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_dataset = Dataset.from_list(train_data).map(tokenize, remove_columns=["prompt", "completion"])
eval_dataset = Dataset.from_list(eval_data).map(tokenize, remove_columns=["prompt", "completion"])

print(f"Tokenized {len(train_dataset)} train / {len(eval_dataset)} eval examples")
sample = train_dataset[0]
masked = sum(1 for l in sample["labels"] if l == -100)
print(f"Sample tokens: {len(sample['input_ids'])} total, {masked} masked, {len(sample['input_ids']) - masked} trained")

## 3.4 Apply LoRA

Identical setup to Chapter 2. LoRA configuration doesn't change with task complexity — that's the beauty of it.

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

## 3.5 Train

JSON generation is harder than single-label classification, so we train longer (15 epochs).

In [ ]:
def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append(x["attention_mask"] + [0] * pad_len)
        labels.append(x["labels"] + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_mask),
        "labels": torch.tensor(labels),
    }

training_args = TrainingArguments(
    output_dir="./chapters/03-extraction/checkpoint",
    num_train_epochs=15,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=(device == "cuda"),
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
)

print("Starting training...")
trainer.train()

## 3.6 Evaluate

We test on sentences the model has never seen. For structured output we check two things:
1. **Valid JSON** — did it output parseable JSON?
2. **Correct fields** — do the extracted values match?

In [ ]:
def extract(sentence):
    prompt = f"Extract: {sentence}\nJSON:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    raw = full.split("JSON:")[-1].strip() if "JSON:" in full else full

    # Try to parse JSON; fall back to regex extraction if malformed
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    # Find the first { ... } block
    m = re.search(r"\{.*?\}", raw, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    return {"raw": raw, "error": "Could not parse JSON"}

test_cases = [
    ("Zoe joined Stark Industries as a devops engineer in November 2023",
     {"name": "Zoe", "company": "Stark Industries", "role": "devops engineer", "start_date": "November 2023"}),
    ("In August 2021, Quinn became a data scientist at Cyberdyne",
     {"name": "Quinn", "company": "Cyberdyne", "role": "data scientist", "start_date": "August 2021"}),
    ("Wayne Enterprises hired Vera as a technical writer in February 2024",
     {"name": "Vera", "company": "Wayne Enterprises", "role": "technical writer", "start_date": "February 2024"}),
    ("Sam has been a QA lead at Initech since June 2020",
     {"name": "Sam", "company": "Initech", "role": "QA lead", "start_date": "June 2020"}),
    ("Since March 2022, Tina has worked as a product manager for Massive Dynamic",
     {"name": "Tina", "company": "Massive Dynamic", "role": "product manager", "start_date": "March 2022"}),
]

print("Predictions:\n")
fields_ok = 0
total_fields = 0
for sentence, expected in test_cases:
    result = extract(sentence)
    print(f"  Input:  {sentence}")
    print(f"  Output: {json.dumps(result)}")

    if "error" not in result:
        for key in expected:
            if result.get(key) == expected[key]:
                fields_ok += 1
        total_fields += len(expected)
    print()

if total_fields > 0:
    print(f"Field accuracy: {fields_ok}/{total_fields} ({100*fields_ok/total_fields:.0f}%)")
else:
    print("No valid JSON outputs — model needs more training.")

## 3.7 Save the Adapter

Same as before — the adapter is ~2MB. You can load it alongside any other task adapter for multi-task models.

In [ ]:
adapter_path = "./adapter"
model.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

## 3.8 What Changed from Chapter 2?

| Chapter 2 (Classification) | Chapter 3 (Extraction) |
|---|---|
| Output: 1 word from 4 options | Output: multi-field JSON object |
| `max_new_tokens=3` | `max_new_tokens=60` |
| Extract with `.split()[0]` | Extract with `json.loads()` + regex fallback |
| 400 examples, 8 epochs | 400 examples, 15 epochs |
| 4 output patterns to learn | Hundreds of output variations |

**Key insight:** The fine-tuning technique is identical. The only thing that changed is the **output format** in the training data. The model doesn't need architectural changes to go from classification to structured generation — it just needs examples of the new behavior.

This is why "prompt → completion" is such a powerful training paradigm: the same pipeline works for any task you can express as text-in, text-out.